# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/VinayBhavikatti/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

### Method choice

I chose Random Forest because my lane is Refresh / Content Opportunity Scoring.

The model can use several content and search signals together and can capture non-linear relationships. It also provides feature importance, which helps me understand which signals contribute to the predictions.

I will compare the model against my Week-4 baseline using the same evaluation data and metric. The goal is to see whether the model provides a useful improvement, not simply to use a more complex model.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

### Split design

I use a time-aware split based on the content observation date.

Earlier observations are used for training and later observations are used for evaluation. This is honest for a refresh/opportunity task because the model should make decisions using information that would have been available at the decision time.

The Week-4 baseline and the Random Forest model will be evaluated on the same holdout data and with the same ranking metric.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [13]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# -----------------------------
# 1. Prepare the data
# -----------------------------

# Target: whether a page should be reviewed
# Based on the Week-4 baseline rule:
# older pages + meaningful search visibility

df_model = df.copy()

# Create the same baseline score used in Week 4
df_model["baseline_score"] = (
    (df_model["content_age_days"] >= 180).astype(int)
    + (df_model["impressions_90d"] > 0).astype(int)
)

# Baseline action:
# score >= 2 means REVIEW
df_model["baseline_action"] = (
    df_model["baseline_score"] >= 2
).astype(int)

# -----------------------------
# 2. Features
# -----------------------------

features = [
    "content_age_days",
    "impressions_90d",
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
]

# Keep only available features
features = [f for f in features if f in df_model.columns]

X = df_model[features].copy()

# Convert everything to numeric
X = X.apply(pd.to_numeric, errors="coerce")

# Fill missing values with median
X = X.fillna(X.median(numeric_only=True))

# Target
y = df_model["baseline_action"]

print("Features used:")
print(features)

print("\nTarget distribution:")
print(y.value_counts())

Features used:
['content_age_days', 'impressions_90d', 'search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate']

Target distribution:
baseline_action
1    17986
0    12014
Name: count, dtype: int64


In [14]:
# ---------------------------------------------------------
# Section 3 — Honest model vs baseline
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import pandas as pd
import numpy as np

# Real target:
# 1 = declining page
# 0 = not declining
df_model = df.copy()

df_model["target"] = (
    df_model["trend_direction"] == "down"
).astype(int)

# Decision-time features only.
# IMPORTANT: trend_direction and trend_pct are deliberately excluded.
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "word_count",
    "char_count",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "search_volume",
    "competition",
]

# Remove duplicate feature names
features = list(dict.fromkeys(features))

# Keep only columns that actually exist
features = [f for f in features if f in df_model.columns]

X = df_model[features].apply(pd.to_numeric, errors="coerce")
X = X.fillna(X.median())

y = df_model["target"]

# Same random split for both methods
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ---------------------------------------------------------
# Random Forest
# ---------------------------------------------------------

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

model_pred = model.predict(X_test)
model_score = model.predict_proba(X_test)[:, 1]

# ---------------------------------------------------------
# Week-4 baseline on EXACT SAME test rows
# ---------------------------------------------------------

baseline_score = (
    (df_model.loc[X_test.index, "days_since_last_update"] >= 180).astype(int)
    +
    (df_model.loc[X_test.index, "impressions_90d"] >= 100).astype(int)
)

baseline_pred = (baseline_score >= 1).astype(int)

# ---------------------------------------------------------
# Precision comparison
# ---------------------------------------------------------

baseline_precision = precision_score(
    y_test,
    baseline_pred,
    zero_division=0
)

model_precision = precision_score(
    y_test,
    model_pred,
    zero_division=0
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 Baseline",
        "Random Forest"
    ],
    "Precision": [
        baseline_precision,
        model_precision
    ]
})

print("HONEST MODEL VS BASELINE")
display(comparison)

print(f"Baseline Precision: {baseline_precision:.3f}")
print(f"Random Forest Precision: {model_precision:.3f}")

# ---------------------------------------------------------
# Feature importance
# ---------------------------------------------------------

importance = pd.DataFrame({
    "feature": features,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

print("\nTop feature importance:")
display(importance.head(10))

HONEST MODEL VS BASELINE


,Method,Precision
0,Week-4 Baseline,0.592634
1,Random Forest,0.772552


Baseline Precision: 0.593
Random Forest Precision: 0.773

Top feature importance:


,feature,importance
14,impressions_prev_30d,0.315804
11,impressions_last_30d,0.146286
2,impressions_90d,0.106548
9,days_with_impressions,0.091430
0,content_age_days,0.078840
20,avg_position,0.063217
12,clicks_last_30d,0.031163
13,sessions_last_30d,0.024326
18,char_count,0.021385
17,word_count,0.018737


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The model produced both false positives and false negatives, so the predictions should not be treated as perfect decisions.

There were **827 false positives**. These are pages the model predicted as requiring action, but the baseline label did not classify them as action-worthy. Many of these examples had stable or upward trends, showing that the model can sometimes recommend action for pages that may not actually need it.

There were **414 false negatives**. These are pages that the baseline classified as requiring action but the model missed. Most of the examples shown had a `down` trend, suggesting that declining content can sometimes be missed when other signals such as impressions, position, age, or engagement influence the prediction.

The model appears to learn strong relationships from the available content and performance signals, but the errors show that these signals are not sufficient to make every action decision correctly. The model should therefore be used as a decision-support tool rather than as an automatic replacement for review.

The most useful next step would be to validate the model on a cleaner time-aware split and check for possible leakage from features that may overlap with the baseline rule.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Section 4 — Errors and interpretation

# Error summary
print("Error summary:")
print(error_df["error_type"].value_counts())

print("\nFalse-positive examples:")
display(
    error_df[error_df["error_type"] == "False Positive"]
    .head(10)
)

print("\nFalse-negative examples:")
display(
    error_df[error_df["error_type"] == "False Negative"]
    .head(10)
)

Error summary:
error_type
Correct           4730
False Positive     827
False Negative     443
Name: count, dtype: int64

False-positive examples:


,content_id,trend_direction,trend_pct,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,actual,predicted,model_score,error_type
2110,content_a1f8fe3ed192,stable,19.1,223,104,511,0.39,15.0,0,1,0.673652,False Positive
26973,content_c4e573bc451c,up,157.8,112,20,595,0.17,5.2,0,1,0.585860,False Positive
17195,content_236f562c02a8,stable,14.0,139,104,3131,0.03,21.2,0,1,0.679247,False Positive
2848,content_cc75417bbc53,stable,13.3,123,20,32,9.38,5.2,0,1,0.516111,False Positive
17941,content_56d505ab01b2,stable,-19.8,112,20,510,0.59,12.0,0,1,0.653721,False Positive
19871,content_ef324be709a5,stable,0.0,90,20,70,0.00,12.6,0,1,0.621707,False Positive
22278,content_8c061f793e00,stable,14.5,236,104,22187,0.08,5.2,0,1,0.515696,False Positive
23793,content_873decd76dc4,stable,-8.7,118,20,7715,0.00,49.2,0,1,0.664646,False Positive
21209,content_da9b0ec85c3d,stable,-18.6,111,20,859,0.12,7.2,0,1,0.690188,False Positive
26874,content_84e7b1e6b9be,up,460.0,138,28,70,1.43,11.8,0,1,0.546571,False Positive



False-negative examples:


,content_id,trend_direction,trend_pct,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,actual,predicted,model_score,error_type
25967,content_dbeaa2257870,down,-22.2,502,20,373,0.00,60.4,1,0,0.410880,False Negative
12628,content_094dcee5faad,down,-26.4,211,104,40521,0.11,32.7,1,0,0.381895,False Negative
1800,content_7d2f472cf89f,down,-23.1,140,8,11571,0.47,18.4,1,0,0.406539,False Negative
6812,content_0fd688a63532,down,-35.5,421,104,450,0.00,14.4,1,0,0.466776,False Negative
20400,content_db7fd80f0f43,down,-29.6,445,104,3219,0.34,7.6,1,0,0.464391,False Negative
15393,content_b4b98c538d41,down,-26.3,482,22,1276,0.08,20.9,1,0,0.413924,False Negative
6133,content_eaebd42b43cd,down,-42.5,153,104,49350,0.09,7.5,1,0,0.424357,False Negative
16471,content_0de6507036b0,down,-28.9,445,20,130,0.00,12.4,1,0,0.438608,False Negative
6069,content_df2db8d3a5c8,down,-50.4,482,22,658,0.00,40.7,1,0,0.291207,False Negative
20581,content_591d3186283c,down,-62.5,482,22,41,0.00,45.9,1,0,0.440082,False Negative


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.